# Baseline Evaluation

To check:

- is the evaluate function below filling in Nans?
- what is the meaning of the forecast_fn?
- why continue when n == 0?

## Imports

In [ ]:
import sys; sys.path.append("..")

import pandas as pd
import numpy as np

from src.rolling_origin_cv import rolling_origin_cutoffs
from src.models import seasonal_naive_forecast, ets_forecast
from src.evaluate import rmsle, wape, score_table

In [ ]:
df = pd.read_parquet("../data/processed/train.parquet")
print(f"{len(df)} rows | {df.date.min().date()} -> {df.date.max().date()}")

In [ ]:
# Sanity check for rolling_origin_cutoffs()

for c in rolling_origin_cutoffs(df.date, horizon=16, n_folds=4):
    tr = (df.date <= c).sum()
    te = ((df.date > c) & (df.date <= c + pd.Timedelta(days=16))).sum()
    print(c.date(), "train rows:", f"{tr:,}", "test rows:", f"{te:,}")
    assert df[df.date <= c].date.max() < (df[(df.date > c)].date.min()), "LEAK: train overlaps test"

In [ ]:
def evaluate_baseline(df, forecast_fn, horizon=16, n_folds=4):
    """Run a forecast function through rolling_origin_cv and return long predictions in a dataframe."""

    cutoffs = rolling_origin_cutoffs(df.date, horizon, n_folds)
    rows = []

    for fold, cutoff in enumerate(cutoffs):
        train = df[df.date <= cutoff]
        test = df[(df.date > cutoff) & (df.date <= cutoff + pd.Timedelta(days=horizon))]

        for (store, family), g_train in train.groupby(["store_nbr", "family"], observed=True):
            s = g_train.set_index("date")["sales"].sort_index().asfreq("D").fillna(0)
            preds = forecast_fn(s, horizon)

            if n == 0:
                continue
            rows.append(pd.DataFrame({
                "fold": fold, "store_nbr": store, "family": family,
                "date": g_test.date.values[:n],
                "y_true": g_test.sales.values[:n],
                "y_pred": preds[:n]
            }))
    
    return pd.concat(rows, ignore_index=True)

In [ ]:
sn_preds = evaluate_baseline(df, seasonal_naive_forecast)
ets_preds = evaluate_baseline(df, ets_forecast)

sn_overall, sn_fam = score_table(sn_preds, "seasonal_naive")
ets_overall, ets_fam = score_table(ets_preds, "ets")

results = pd.concat([sn_overall, ets_overall], ignore_index=True)
print(results)

In [ ]:
results.to_csv("../outputs/results/baseline_scores.csv", index=False)
sn_fam.to_csv("../outputs/results/baseline_scores_by_family.csv", index=False)